### EMBEDDING AND VECTOR DB

In [ ]:
# Import libraries for embedding generation, vector storage, and array operations.
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
# Define file paths, collection name, and embedding model used throughout the pipeline.
CHUNKS_PATH     = "../data/processed/chunks.json"
CHROMA_PATH     = "../data/processed/chroma_db"
COLLECTION_NAME = "elte_ik"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

In [ ]:
# Wrap SentenceTransformer in a class to load the model once and encode text batches.
class EmbeddingPipeline:
    def __init__(self, model_name: str = EMBEDDING_MODEL):
        self.model = SentenceTransformer(model_name)
        print(f"Loaded model: {model_name}")

    def encode(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(texts, show_progress_bar=True)

In [ ]:
# Load chunks from disk and generate a 384-dim embedding vector for each one.
with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
print(f"Loaded {len(chunks)} chunks")

pipeline = EmbeddingPipeline()
texts = [c["content"] for c in chunks]
embeddings = pipeline.encode(texts)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Create (or open) the ChromaDB collection and upsert all embeddings in batches of 5000.
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

ids = [str(c["metadata"]["chunk_id"]) for c in chunks]
embs = embeddings.tolist()
metas = [c["metadata"] for c in chunks]

BATCH_SIZE = 5000
for i in range(0, len(ids), BATCH_SIZE):
    collection.upsert(
        ids=ids[i:i+BATCH_SIZE],
        embeddings=embs[i:i+BATCH_SIZE],
        documents=texts[i:i+BATCH_SIZE],
        metadatas=metas[i:i+BATCH_SIZE],
    )
print(f"Upserted {collection.count()} documents into '{COLLECTION_NAME}'")

In [ ]:
# Run a sample query to verify retrieval returns relevant chunks from the collection.
query = "what is Notification of accommodation?"
query_emb = pipeline.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=3)
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n--- Result {i+1} (chunk {meta['chunk_id']}, {meta['file_name']}) ---")
    print(doc[:300])